In [75]:
from timeit import timeit
import altair as alt
import pandas as pd

alt.data_transformers.disable_max_rows()

n = 100000
batch_size = 100

def make_lists(n):
    return [list(range(i)) for i in range(batch_size, n+batch_size, 50)]

funcs = {
    "pop_last": lambda l: l.pop(),
    "pop_first": lambda l: l.pop(0),
    "insert_first": lambda l: l.insert(0, None),
    "append": lambda l: l.append(None)
}

df = pd.DataFrame({
    k: list(map(lambda l: timeit(lambda: v(l), number=batch_size), make_lists(n)))
    for k,v in funcs.items()
})

df = df.rolling(batch_size, center=True).mean().dropna()
df = df.reset_index().melt(id_vars="index", value_vars=funcs.keys())

alt.Chart(df.reset_index()).mark_line().encode(
    x="index",
    y="value",
    color="variable"
)

alt.Chart(...)